In [10]:
import llava
import transformers
import json

In [ ]:
proj_dir = "/home/yifayang/Documents/Projects/Other/LLaVA"

class ModelArguments:
    model_name_or_path = f"{proj_dir}/playground/checkpoints/vicuna-7b-v1.3"

class TrainingArguments:
    cache_dir = "/home/yifayang/.cache/huggingface"
    model_max_length = 2048

class DataArguments:
    data_path = f"{proj_dir}/playground/data/finetune-instruct-150k/llava_instruct_150k.json"

model_args = ModelArguments()
training_args = TrainingArguments()
data_args = DataArguments()

In [7]:
tokenizer = transformers.AutoTokenizer.from_pretrained(
    model_args.model_name_or_path,
    cache_dir=training_args.cache_dir,
    model_max_length=training_args.model_max_length,
    padding_side="right",
    use_fast=False,
)

You are using the legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This means that tokens that come after special tokens will not be properly handled. We recommend you to read the related pull request available at https://github.com/huggingface/transformers/pull/24565


In [25]:
list_data_dict = json.load(open(f"{proj_dir}/playground/data/finetune-instruct-150k/llava_instruct_150k.json", "r"))

In [28]:
list_data_dict[0]

{'id': '000000033471',
 'image': '000000033471.jpg',
 'conversations': [{'from': 'human',
   'value': '<image>\nWhat are the colors of the bus in the image?'},
  {'from': 'gpt', 'value': 'The bus in the image is white and red.'},
  {'from': 'human',
   'value': 'What feature can be seen on the back of the bus?'},
  {'from': 'gpt', 'value': 'The back of the bus features an advertisement.'},
  {'from': 'human',
   'value': 'Is the bus driving down the street or pulled off to the side?'},
  {'from': 'gpt',
   'value': 'The bus is driving down the street, which is crowded with people and other vehicles.'}]}

```bash
deepspeed --include localhost:0 llava/train/train_mem.py \
    --deepspeed ./scripts/zero3.json \
    --lora_enable True \
    --model_name_or_path ./playground/checkpoints/vicuna-7b-v1.3 \
    --version v1 \
    --data_path ./playground/data/finetune-instruct-150k/llava_instruct_150k.json \
    --image_folder ./playground/data/coco/train2017 \
    --vision_tower openai/clip-vit-large-patch14 \
    --pretrain_mm_mlp_adapter playground/checkpoints/llava-vicuna-7b-v1.3-pretrain/mm_projector.bin \
    --mm_vision_select_layer -2 \
    --mm_use_im_start_end False \
    --mm_use_im_patch_token False \
    --bf16 True \
    --output_dir ./playground/checkpoints/llava-vicuna-7b-v1.3-finetune_lora \
    --num_train_epochs 1 \
    --per_device_train_batch_size 16 \
    --per_device_eval_batch_size 4 \
    --gradient_accumulation_steps 1 \
    --evaluation_strategy "no" \
    --save_strategy "steps" \
    --save_steps 50000 \
    --save_total_limit 1 \
    --learning_rate 2e-5 \
    --weight_decay 0. \
    --warmup_ratio 0.03 \
    --lr_scheduler_type "cosine" \
    --logging_steps 1 \
    --tf32 True \
    --model_max_length 2048 \
    --gradient_checkpointing True \
    --lazy_preprocess True \
    --dataloader_num_workers 4 \
    --report_to wandb
```

```bash

python scripts/merge_lora_weights.py \
    --model-base playground/checkpoints/vicuna-7b-v1.3 \
    --model-path playground/checkpoints/llava-vicuna-7b-v1.3-finetune_lora \
    --save-model-path playground/checkpoints/llava-vicuna-7b-v1.3-finetune_lora_merged
```
```bash
python -m llava.serve.cli \
    --model-path playground/checkpoints/llava-vicuna-7b-v1.3-finetune_lora_merged \
    --image-file "https://llava-vl.github.io/static/images/view.jpg" \
    --load-4bit
```